# Road Segmentation — Multi-Model Training & Comparison

This notebook trains **all model configurations** automatically and produces a comparison dashboard to help select the best architecture.

### What it does
1. Installs dependencies, clones the repo, downloads the dataset
2. Runs training for each config in `configs/` (U-Net, DeepLabV3+, LinkNet, SegFormer)
3. Collects all results and produces:
   - Side-by-side metrics comparison table
   - Overlaid training curves (loss, IoU)
   - Prediction comparison grid across all models
   - Final recommendation
4. Saves all checkpoints and results to Google Drive (optional)

### How to run on Colab
1. `Runtime > Change runtime type > T4 GPU`
2. Set `REPO_URL` with your GitHub token (for private repos)
3. Set Kaggle credentials in the dataset cell
4. **Run All** (`Runtime > Run all`) — everything is automated

---
## 1. Setup & Installation

In [17]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

# ====== SET YOUR REPO URL (use token for private repos) ======
REPO_URL = "https://github.com/minaessam2015/road-segmentation.git"
# REPO_URL = "https://<TOKEN>@github.com/minaessam2015/road-segmentation.git"

if IN_COLAB:
    REPO_DIR = "/content/road-segmentation"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        print(f"Cloned to {REPO_DIR}")
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
        print(f"Pulled latest to {REPO_DIR}")

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub", "pyyaml", "tqdm"], check=True)
    print("Dependencies installed.")
else:
    REPO_DIR = None
    print("Running locally.")

Pulled latest to /content/road-segmentation
Dependencies installed.


In [18]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

if IN_COLAB:
    PROJECT_ROOT = Path(REPO_DIR).resolve()
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: /content/road-segmentation


---
## 2. Device Detection

In [19]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple MPS")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla T4
Memory: 15.6 GB
Device: cuda
PyTorch: 2.10.0+cu128


---
## 3. Download Dataset

In [20]:
from road_segmentation.paths import DEEPGLOBE_DATASET_DIR

train_dir = DEEPGLOBE_DATASET_DIR / "train"
has_images = train_dir.exists() and any(train_dir.glob("*_sat.*"))

if not has_images:
    if IN_COLAB:
        kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
        if not kaggle_json.exists():
            # Option A: Upload kaggle.json
            print("Upload your kaggle.json file:")
            from google.colab import files
            uploaded = files.upload()
            kaggle_json.parent.mkdir(parents=True, exist_ok=True)
            kaggle_json.write_bytes(list(uploaded.values())[0])
            kaggle_json.chmod(0o600)

        # Option B: Set credentials directly (uncomment):
        # os.environ["KAGGLE_USERNAME"] = "your_username"
        # os.environ["KAGGLE_KEY"] = "your_api_key"

    print("Downloading DeepGlobe dataset...")
    from road_segmentation.data.download import download_dataset
    download_dataset()

sat_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_sat.*")))
mask_count = len(list((DEEPGLOBE_DATASET_DIR / "train").glob("*_mask.*")))
print(f"Training: {sat_count} images, {mask_count} masks")

Training: 6226 images, 6226 masks


---
## 4. Experiment Settings

Choose which configs to run and set shared overrides (subset size, epochs, etc.).

In [25]:
# ====== CONFIGS TO COMPARE ======
# Comment out any you want to skip
CONFIGS_TO_RUN = [
    "unet_resnet34.yaml",
    "deeplabv3plus_resnet50.yaml",
    "linknet_resnet34.yaml",
    "segformer_mitb2.yaml",  # uncomment if you have enough GPU memory
]

# ====== SHARED OVERRIDES ======
# These apply to ALL experiments for a fair comparison
SHARED_OVERRIDES = {
    "data.subset_size": 1000,   # use a subset for quick experiments
    "training.epochs": 15,      # reduce epochs for speed
}

# Auto-adjust for environment
if device.type == "cpu":
    SHARED_OVERRIDES.setdefault("data.subset_size", 200)
    SHARED_OVERRIDES.setdefault("training.epochs", 5)
    SHARED_OVERRIDES.setdefault("data.num_workers", 0)
    SHARED_OVERRIDES.setdefault("data.batch_size", 4)
    SHARED_OVERRIDES.setdefault("training.mixed_precision", False)

if IN_COLAB:
    SHARED_OVERRIDES.setdefault("data.num_workers", 2)

print(f"Experiments to run: {len(CONFIGS_TO_RUN)}")
for c in CONFIGS_TO_RUN:
    print(f"  - {c}")
if SHARED_OVERRIDES:
    print(f"\nShared overrides:")
    for k, v in SHARED_OVERRIDES.items():
        print(f"  {k} = {v}")

Experiments to run: 4
  - unet_resnet34.yaml
  - deeplabv3plus_resnet50.yaml
  - linknet_resnet34.yaml
  - segformer_mitb2.yaml

Shared overrides:
  data.subset_size = 1000
  training.epochs = 15
  data.num_workers = 2


---
## 5. Run All Experiments

This cell loops through each config, trains the model, and collects results.

In [26]:
import gc
import random
import time

import numpy as np
import pandas as pd

from road_segmentation.config import load_config
from road_segmentation.data.eda import discover_image_mask_pairs
from road_segmentation.data.split import split_pairs
from road_segmentation.data.transforms import get_train_transform, get_val_transform
from road_segmentation.data.dataset import create_dataloaders
from road_segmentation.models.factory import create_model, count_parameters, freeze_encoder
from road_segmentation.training.losses import create_loss
from road_segmentation.training.trainer import Trainer

# Collect results across experiments
all_results = {}  # config_name -> {"config", "trainer", "best_metrics", "history", "train_time"}

for config_name in CONFIGS_TO_RUN:
    print("\n" + "=" * 70)
    print(f"  EXPERIMENT: {config_name}")
    print("=" * 70)

    # --- Load config with overrides ---
    config = load_config(PROJECT_ROOT / "configs" / config_name)
    for key, value in SHARED_OVERRIDES.items():
        parts = key.split(".")
        obj = config
        for part in parts[:-1]:
            obj = getattr(obj, part)
        setattr(obj, parts[-1], value)

    # --- Seed ---
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)

    # --- Data (same split for all experiments) ---
    pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
    train_pairs, val_pairs = split_pairs(
        pairs,
        val_ratio=config.data.val_split_ratio,
        seed=config.data.val_split_seed,
        subset_size=config.data.subset_size,
    )

    train_transform = get_train_transform(
        image_size=config.data.image_size,
        aug_steps=config.augmentations.train,
        mean=config.normalization.mean,
        std=config.normalization.std,
    )
    val_transform = get_val_transform(
        image_size=config.data.image_size,
        mean=config.normalization.mean,
        std=config.normalization.std,
    )
    train_loader, val_loader = create_dataloaders(
        train_pairs, val_pairs,
        train_transform, val_transform,
        batch_size=config.data.batch_size,
        num_workers=config.data.num_workers,
        pin_memory=config.data.pin_memory and device.type == "cuda",
    )

    # --- Model ---
    model = create_model(
        arch=config.model.arch,
        encoder_name=config.model.encoder_name,
        encoder_weights=config.model.encoder_weights,
        in_channels=config.model.in_channels,
        classes=config.model.classes,
        **config.model.decoder_kwargs,
    )
    if config.training.freeze_encoder_epochs > 0:
        freeze_encoder(model)

    params = count_parameters(model)
    print(f"Model: {config.model.arch} + {config.model.encoder_name} | {params['total']/1e6:.1f}M params")

    # --- Train ---
    loss_fn = create_loss(config.loss.type, config.loss.params)
    trainer = Trainer(config, model, train_loader, val_loader, loss_fn, device)

    t0 = time.time()
    best_metrics = trainer.train()
    train_time = time.time() - t0

    # --- Collect results ---
    label = f"{config.model.arch}+{config.model.encoder_name}"
    all_results[label] = {
        "config_name": config_name,
        "config": config,
        "trainer": trainer,
        "best_metrics": best_metrics,
        "history": trainer.history,
        "train_time_min": train_time / 60,
        "total_params_m": params["total"] / 1e6,
    }

    print(f"\nDone: val_iou={best_metrics.get('val_iou', 0):.4f} | time={train_time/60:.1f} min")

    # Free GPU memory before next experiment
    del model, trainer, loss_fn, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print(f"  ALL {len(all_results)} EXPERIMENTS COMPLETE")
print("=" * 70)


  EXPERIMENT: unet_resnet34.yaml
Model: Unet + resnet34 | 24.4M params



Done: val_iou=0.5279 | time=10.0 min

  EXPERIMENT: deeplabv3plus_resnet50.yaml


Model: DeepLabV3Plus + resnet50 | 26.7M params



Done: val_iou=0.4843 | time=10.3 min

  EXPERIMENT: linknet_resnet34.yaml
Model: Linknet + resnet34 | 21.8M params



Done: val_iou=0.5066 | time=9.2 min

  EXPERIMENT: segformer_mitb2.yaml


Model: Segformer + mit_b2 | 24.7M params



Done: val_iou=0.4853 | time=14.6 min

  ALL 4 EXPERIMENTS COMPLETE


---
## 6. Comparison Dashboard

### 6a. Metrics Comparison Table

In [27]:
import pandas as pd

rows = []
for label, result in all_results.items():
    bm = result["best_metrics"]
    rows.append({
        "Model": label,
        "Best IoU": bm.get("val_iou", 0),
        "Best Dice": bm.get("val_dice", 0),
        "Precision": bm.get("val_precision", 0),
        "Recall": bm.get("val_recall", 0),
        "Val Loss": bm.get("val_loss", 0),
        "Params (M)": result["total_params_m"],
        "Time (min)": result["train_time_min"],
    })

comparison_df = pd.DataFrame(rows).sort_values("Best IoU", ascending=False)
comparison_df = comparison_df.reset_index(drop=True)
comparison_df.index += 1  # rank starting from 1
comparison_df.index.name = "Rank"

print("Model Comparison (ranked by IoU)")
print("=" * 70)
display(comparison_df.round(4))

winner = comparison_df.iloc[0]["Model"]
print(f"\nBest model: {winner} (IoU={comparison_df.iloc[0]['Best IoU']:.4f})")

Model Comparison (ranked by IoU)


,Model,Best IoU,Best Dice,Precision,Recall,Val Loss,Params (M),Time (min)
Rank,,,,,,,,
1,Unet+resnet34,0.5279,0.6910,0.6696,0.7139,0.2161,24.4364,10.0053
2,Linknet+resnet34,0.5066,0.6725,0.6721,0.6729,0.2296,21.7719,9.2118
3,Segformer+mit_b2,0.4853,0.6534,0.6151,0.6968,0.2672,24.7224,14.6225
4,DeepLabV3Plus+resnet50,0.4843,0.6526,0.6604,0.6449,0.2448,26.6776,10.2578



Best model: Unet+resnet34 (IoU=0.5279)


### 6b. Training Curves — All Models Overlaid

In [28]:
import matplotlib.pyplot as plt

plt.style.use("ggplot")
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

for label, result in all_results.items():
    history = result["history"]
    epochs = [h["epoch"] for h in history]

    axes[0, 0].plot(epochs, [h["train_loss"] for h in history], label=f"{label} (train)", linewidth=1.5)
    axes[0, 0].plot(epochs, [h["val_loss"] for h in history], label=f"{label} (val)", linestyle="--", linewidth=1.5)

    axes[0, 1].plot(epochs, [h["val_iou"] for h in history], label=label, linewidth=2)

    axes[1, 0].plot(epochs, [h["val_dice"] for h in history], label=label, linewidth=2)

    axes[1, 1].plot(epochs, [h["lr"] for h in history], label=label, linewidth=1.5)

axes[0, 0].set_title("Loss", fontsize=13)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True)

axes[0, 1].set_title("Validation IoU", fontsize=13)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True)

axes[1, 0].set_title("Validation Dice", fontsize=13)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True)

axes[1, 1].set_title("Learning Rate", fontsize=13)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_yscale("log")
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True)

fig.suptitle("Training Curves — All Models", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### 6c. Prediction Comparison — Same Samples Across All Models

In [29]:
from road_segmentation.training.visualization import denormalize_image

# Load best checkpoint for each model and predict on the same val samples
N_SAMPLES = 4
n_models = len(all_results)

# Get a fixed val batch from the last experiment's config
last_config = list(all_results.values())[-1]["config"]
pairs = discover_image_mask_pairs(DEEPGLOBE_DATASET_DIR)
_, val_pairs = split_pairs(
    pairs,
    val_ratio=last_config.data.val_split_ratio,
    seed=last_config.data.val_split_seed,
    subset_size=last_config.data.subset_size,
)
val_transform = get_val_transform(
    image_size=last_config.data.image_size,
    mean=last_config.normalization.mean,
    std=last_config.normalization.std,
)
from road_segmentation.data.dataset import RoadSegmentationDataset
val_ds = RoadSegmentationDataset(val_pairs[:N_SAMPLES], transform=val_transform)
samples = [val_ds[i] for i in range(N_SAMPLES)]
images = torch.stack([s["image"] for s in samples])
masks_gt = torch.stack([s["mask"] for s in samples])

# columns: Input | GT | model1 pred | model2 pred | ...
n_cols = 2 + n_models
fig, axes = plt.subplots(N_SAMPLES, n_cols, figsize=(4 * n_cols, 4 * N_SAMPLES))
if N_SAMPLES == 1:
    axes = axes[np.newaxis, :]

mean = last_config.normalization.mean
std = last_config.normalization.std

for i in range(N_SAMPLES):
    # Input image
    img = denormalize_image(images[i], mean, std)
    axes[i, 0].imshow(img)
    axes[i, 0].axis("off")
    if i == 0:
        axes[i, 0].set_title("Input", fontsize=11, fontweight="bold")

    # Ground truth
    axes[i, 1].imshow(masks_gt[i, 0].numpy(), cmap="gray")
    axes[i, 1].axis("off")
    if i == 0:
        axes[i, 1].set_title("Ground Truth", fontsize=11, fontweight="bold")

for col_idx, (label, result) in enumerate(all_results.items()):
    # Load best model
    cfg = result["config"]
    model = create_model(
        arch=cfg.model.arch,
        encoder_name=cfg.model.encoder_name,
        encoder_weights=None,  # load from checkpoint, not imagenet
        in_channels=cfg.model.in_channels,
        classes=cfg.model.classes,
    )
    ckpt_dir = Path(cfg.checkpoint.save_dir) / cfg.logging.experiment_name
    best_ckpt = ckpt_dir / "best.pth"
    if best_ckpt.exists():
        state = torch.load(best_ckpt, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model_state_dict"])
    model.eval().to(device)

    with torch.no_grad():
        logits = model(images.to(device))
        preds = (torch.sigmoid(logits) >= 0.5).cpu().numpy()

    iou = result["best_metrics"].get("val_iou", 0)
    for i in range(N_SAMPLES):
        axes[i, 2 + col_idx].imshow(preds[i, 0], cmap="gray")
        axes[i, 2 + col_idx].axis("off")
        if i == 0:
            axes[i, 2 + col_idx].set_title(f"{label}\nIoU={iou:.3f}", fontsize=10, fontweight="bold")

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fig.suptitle("Prediction Comparison Across Models", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6d. Efficiency Plot — IoU vs Training Time vs Model Size

In [30]:
fig, ax = plt.subplots(figsize=(10, 6))

for label, result in all_results.items():
    iou = result["best_metrics"].get("val_iou", 0)
    time_min = result["train_time_min"]
    params_m = result["total_params_m"]

    # Bubble size proportional to model size
    ax.scatter(time_min, iou, s=params_m * 15, alpha=0.7, edgecolors="black", linewidth=1)
    ax.annotate(label, (time_min, iou), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

ax.set_xlabel("Training Time (minutes)", fontsize=12)
ax.set_ylabel("Best Validation IoU", fontsize=12)
ax.set_title("Efficiency: IoU vs Training Time (bubble size = params)", fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Full Training History

In [31]:
for label, result in all_results.items():
    print(f"\n{label}")
    print("-" * 50)
    df = pd.DataFrame(result["history"])
    display(df.round(4))


Unet+resnet34
--------------------------------------------------


,epoch,train_loss,val_loss,val_iou,val_dice,val_precision,val_recall,lr,epoch_time_s
0,0,0.5118,0.3525,0.3380,0.5052,0.5424,0.4728,0.0010,36.3770
1,1,0.3440,0.3036,0.3879,0.5590,0.5961,0.5263,0.0010,35.2267
2,2,0.3173,0.3115,0.3916,0.5628,0.4795,0.6809,0.0010,34.9249
3,3,0.3088,0.2751,0.4300,0.6014,0.6091,0.5940,0.0009,36.2589
4,4,0.2995,0.2863,0.4242,0.5957,0.5166,0.7034,0.0008,36.2476
5,5,0.2976,0.2609,0.4550,0.6255,0.6174,0.6338,0.0010,37.9275
6,6,0.2744,0.2566,0.4552,0.6256,0.6924,0.5705,0.0010,38.7596
7,7,0.2655,0.2524,0.4709,0.6403,0.5893,0.7009,0.0009,37.8450
8,8,0.2598,0.2433,0.4808,0.6494,0.6336,0.6659,0.0008,37.7339
9,9,0.2436,0.2297,0.5050,0.6711,0.6502,0.6934,0.0007,39.3405



DeepLabV3Plus+resnet50
--------------------------------------------------


,epoch,train_loss,val_loss,val_iou,val_dice,val_precision,val_recall,lr,epoch_time_s
0,0,0.5122,0.4028,0.2888,0.4482,0.4014,0.5074,0.0005,38.9068
1,1,0.3968,0.3555,0.3057,0.4682,0.5411,0.4127,0.0005,34.3368
2,2,0.3668,0.3375,0.3297,0.4959,0.5648,0.4420,0.0005,33.1971
3,3,0.3430,0.3073,0.3725,0.5428,0.6024,0.4938,0.0005,38.2736
4,4,0.3167,0.2972,0.3899,0.5611,0.6742,0.4804,0.0005,37.9929
5,5,0.2958,0.2743,0.4395,0.6106,0.5910,0.6316,0.0005,38.9410
6,6,0.2890,0.2759,0.4271,0.5985,0.6749,0.5377,0.0004,38.4673
7,7,0.2760,0.2629,0.4549,0.6253,0.6428,0.6088,0.0004,37.9262
8,8,0.2703,0.2580,0.4593,0.6295,0.6657,0.5971,0.0003,39.8959
9,9,0.2644,0.2548,0.4654,0.6352,0.6840,0.5929,0.0003,40.6856



Linknet+resnet34
--------------------------------------------------


,epoch,train_loss,val_loss,val_iou,val_dice,val_precision,val_recall,lr,epoch_time_s
0,0,0.6771,0.5797,0.1733,0.2954,0.3354,0.2639,0.0010,34.1026
1,1,0.5138,0.4613,0.2907,0.4504,0.4910,0.4160,0.0010,32.8283
2,2,0.4365,0.4057,0.3316,0.4981,0.4270,0.5974,0.0010,32.6039
3,3,0.3935,0.3795,0.3214,0.4864,0.5995,0.4092,0.0009,32.7489
4,4,0.3609,0.3413,0.3703,0.5405,0.5661,0.5171,0.0008,31.3573
5,5,0.3307,0.3005,0.4108,0.5824,0.5162,0.6682,0.0010,34.0653
6,6,0.2985,0.2750,0.4408,0.6119,0.5606,0.6736,0.0010,34.4873
7,7,0.2747,0.2586,0.4607,0.6308,0.6218,0.6401,0.0009,34.1466
8,8,0.2650,0.2504,0.4757,0.6447,0.5971,0.7005,0.0008,33.3106
9,9,0.2573,0.2458,0.4834,0.6517,0.5945,0.7212,0.0007,34.9117



Segformer+mit_b2
--------------------------------------------------


,epoch,train_loss,val_loss,val_iou,val_dice,val_precision,val_recall,lr,epoch_time_s
0,0,0.6846,0.5127,0.2628,0.4162,0.3534,0.5062,0.0001,51.6893
1,1,0.4659,0.3921,0.3649,0.5347,0.5177,0.5527,0.0001,48.8032
2,2,0.3957,0.3870,0.3656,0.5354,0.4170,0.7479,0.0001,48.5730
3,3,0.3600,0.3233,0.4257,0.5972,0.5744,0.6219,0.0001,49.1694
4,4,0.3313,0.3092,0.4447,0.6156,0.5809,0.6547,0.0001,48.2654
5,5,0.3150,0.2983,0.4504,0.6211,0.6297,0.6127,0.0000,48.1841
6,6,0.3013,0.2940,0.4604,0.6305,0.5781,0.6932,0.0000,47.9546
7,7,0.2935,0.2876,0.4675,0.6371,0.5981,0.6816,0.0000,48.9057
8,8,0.2840,0.2759,0.4744,0.6436,0.6072,0.6845,0.0000,48.9137
9,9,0.2760,0.2724,0.4790,0.6477,0.6191,0.6792,0.0000,49.4252


---
## 8. Save Results (Optional)

Save all checkpoints and comparison results to Google Drive.

In [32]:
import shutil

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/road_segmentation")
else:
    save_dir = PROJECT_ROOT / "experiment_results"

save_dir.mkdir(parents=True, exist_ok=True)

# Save comparison table
comparison_df.to_csv(save_dir / "comparison.csv")
print(f"Comparison table saved to: {save_dir / 'comparison.csv'}")

# Save best checkpoint for each model
for label, result in all_results.items():
    cfg = result["config"]
    ckpt_dir = Path(cfg.checkpoint.save_dir) / cfg.logging.experiment_name
    best_ckpt = ckpt_dir / "best.pth"
    if best_ckpt.exists():
        dest = save_dir / f"{label}_best.pth"
        shutil.copy(best_ckpt, dest)
        print(f"  {label} -> {dest}")

print(f"\nAll results saved to: {save_dir}")

ValueError: mount failed